In [ ]:
DATA = 'result_e200_b64_d20.tsv' 

In [1]:
import sentencepiece as spm
import pandas as pd
import numpy as np
import math
import copy

In [2]:
sp = spm.SentencePieceProcessor()
sp.Load('split.model')

True

In [3]:
print(sp.EncodeAsPieces('ちょっとしたデイサービスみたいなのよね。'))

['▁ちょっと', 'した', 'デイ', 'サービス', 'みたいなの', 'よね', '。']


In [213]:
df_ref = pd.read_csv(DATA,sep='\t',header=0,usecols=['target(正解)'])
df_can = pd.read_csv(DATA,sep='\t',header=0,usecols=['output(出力)'])

In [214]:
ref_list = df_ref.values.tolist()
can_list = df_can.values.tolist()

In [215]:
all_sep_ref = []
all_sep_can = []

for ref,can in zip(ref_list,can_list):
    ref = ''.join(map(str,ref)).replace('\n','')
    can = ''.join(map(str,can)).replace('\n','')
    sep_ref = sp.EncodeAsPieces(ref)
    sep_can = sp.EncodeAsPieces(can)
    if sep_ref[0] == '▁':
        del sep_ref[0]
    else:
        sep_ref[0] = sep_ref[0][1:]
    if sep_can[0] == '▁':
        del sep_can[0]
    else:
        sep_can[0] = sep_can[0][1:]
    all_sep_ref.append(sep_ref)
    all_sep_can.append(sep_can)

In [216]:
print(can_list[:5])

[['いや、そうやってる。\n'], ['考えたことないのよ。\n'], ['うん、うん。\n'], ['それで。\n'], ['だから、その、じゃあさー、(うん)まあ、それ代んだけど、(うんうん)だからもうあれは1年で、(うん)なんかそれがあの。\n']]


In [217]:
ngram_match = []
ngram_r = []
ngram_c = []

In [218]:
#candidate,referenceの全piece数c,rとcandidateのUni-gram一致数
uni_sep_ref = copy.deepcopy(all_sep_ref)
uni_sep_can = copy.deepcopy(all_sep_can)
uni_can_match = 0
uni_r = 0
uni_c = 0
for ref,can in zip(uni_sep_ref,uni_sep_can):
    uni_r += len(ref)
    uni_c += len(can)
    for uni_gram in can:
        if uni_gram in ref:
            uni_can_match += 1
            ref.remove(uni_gram)
ngram_match.append(uni_can_match)
ngram_r.append(uni_r)
ngram_c.append(uni_c)
print(uni_r,uni_c,uni_can_match)

24003 27905 4036


In [219]:
#bi-gram一致数
sep_ref = copy.deepcopy(all_sep_ref)
sep_can = copy.deepcopy(all_sep_can)
bi_sep_ref = []
bi_sep_can = []
bi_can_match = 0
bi_r = 0
bi_c = 0
for ref,can in zip(sep_ref,sep_can):
    bi_ref = []
    bi_can = []
    i = 1
    while i != len(ref):
        bi_ref.append(ref[i-1] + ref[i])
        i += 1
    i = 1
    while i != len(can):
        bi_can.append(can[i-1] + can[i])
        i += 1
    bi_sep_ref.append(bi_ref)
    bi_sep_can.append(bi_can)
    
for ref,can in zip(bi_sep_ref,bi_sep_can):
    bi_r += len(ref)
    bi_c += len(can)
    for bi_gram in can:
        if bi_gram in ref:
            bi_can_match += 1
            ref.remove(bi_gram)
ngram_match.append(bi_can_match)
ngram_r.append(bi_r)
ngram_c.append(bi_c)
print(bi_r,bi_c,bi_can_match)

20667 24569 138


In [220]:
print(ngram_c)

[27905, 24569]


In [221]:
Pn = []
for match,c in zip(ngram_match,ngram_c):
    Pn.append(match/c)
if uni_c < uni_r:
    BP = math.exp(1-c/r)
else:
    BP = 1
print(Pn)

[0.14463357821179001, 0.005616834221987057]


In [222]:
gm = 0
for P in Pn:
    gm += math.log(P)
BLEU = BP * math.exp(gm/len(Pn))
print(BLEU)

0.028502330286284027


In [225]:
df1 = pd.read_csv('result_e100_b128_d20.tsv',sep='\t',header=0)
df2 = pd.read_csv('result_e100_b64_d20.tsv',sep='\t',header=0)
df3 = pd.read_csv('result_e200_b128_d20.tsv',sep='\t',header=0)
df4 = pd.read_csv('result_e200_b64_d20.tsv',sep='\t',header=0)

with pd.ExcelWriter('result.xlsx') as writer:
    df1.to_excel(writer,sheet_name='A')
    df2.to_excel(writer,sheet_name='B')
    df3.to_excel(writer,sheet_name='C')
    df4.to_excel(writer,sheet_name='D')